# 02 — Temporal Train / Validation / Test Split

Prepares data for PySpark implicit ALS by:
1. Joining `interactions_dedup` (timestamps) with ID maps (integer IDs)
2. Keeping ALL interactions (shelved, read, rated) — not just explicit ratings
3. Filtering cold-start users/books with < 5 interactions (any type)
4. Splitting temporally: **train (70%) → val (20%) → test (10%)**
5. Saving splits as Parquet to GCS with schema: `(user_id: int, book_id: int, is_read: int, rating: int)`

The interaction strength `r = 1 + 2*is_read + β*max(0, rating-3)` is NOT computed here
because β is a tunable hyperparameter set during ALS training (notebook 03).

In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.expanduser(
    "~/.config/gcloud/application_default_credentials.json"
)

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("goodreads_data_split")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.gs.auth.type", "APPLICATION_DEFAULT")
    .getOrCreate()
)

GCS_BASE = "gs://nshen7-personal-bucket/projects/rec_sys_goodreads"
PARQUET_BASE = f"{GCS_BASE}/data/parquet"
OUTPUT_BASE = f"{GCS_BASE}/data/splits_sample_mode"

# --- Toggle: set to False for full dataset (e.g., on Dataproc) ---
SAMPLE_MODE = True
SAMPLE_FRACTION = 0.005  # ~0.5% of data (~1.1M interactions)
SAMPLE_SEED = 42

print(f"SAMPLE_MODE: {SAMPLE_MODE}" + (f" (fraction={SAMPLE_FRACTION})" if SAMPLE_MODE else " (full dataset)"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/12 17:54:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SAMPLE_MODE: True (fraction=0.005)


In [2]:
# Load ALL interactions (every row = a shelved event).
# Do NOT filter to rating > 0 — we now use all interaction types.
interactions_dedup = spark.read.parquet(f"{PARQUET_BASE}/goodreads_interactions_dedup")

if SAMPLE_MODE:
    interactions_dedup = interactions_dedup.sample(
        fraction=SAMPLE_FRACTION, seed=SAMPLE_SEED
    )

user_id_map = spark.read.parquet(f"{PARQUET_BASE}/user_id_map")
book_id_map = spark.read.parquet(f"{PARQUET_BASE}/book_id_map")

print(f"interactions_dedup (all): {interactions_dedup.count():,} rows")
print(f"user_id_map:              {user_id_map.count():,} rows")
print(f"book_id_map:              {book_id_map.count():,} rows")

interactions_dedup (all): 1,142,544 rows
user_id_map:              876,145 rows
book_id_map:              2,360,650 rows


## 1. Join interactions_dedup with ID maps

Goal: get integer `user_id` + `book_id` alongside `is_read`, `rating`, and timestamps from `interactions_dedup`.

In [3]:
# Broadcast the small ID map tables to avoid expensive shuffle joins
# user_id_map (~876K rows) and book_id_map (~2.3M rows) fit easily in memory

# Join with user_id_map: interactions_dedup.user_id (str) -> user_id_csv (int)
df = (
    interactions_dedup
    .join(
        F.broadcast(user_id_map),
        interactions_dedup["user_id"] == user_id_map["user_id"],
        "inner",
    )
    .drop(user_id_map["user_id"])
)

# Join with book_id_map: interactions_dedup.book_id (str->int) -> book_id_csv (int)
df = (
    df
    .withColumn("book_id_int", F.col("book_id").cast("int"))
    .join(
        F.broadcast(book_id_map),
        F.col("book_id_int") == book_id_map["book_id"],
        "inner",
    )
    .drop(book_id_map["book_id"])
)

# Select clean columns — carry is_read and rating through
df = df.select(
    F.col("user_id_csv").alias("user_id"),
    F.col("book_id_csv").alias("book_id"),
    F.col("is_read").cast("int").alias("is_read"),   # boolean -> 0/1
    F.col("rating").cast("int").alias("rating"),       # long -> int (0-5, 0=unrated)
    F.col("date_added"),
)

print(f"Joined dataset: {df.count():,} rows")
df.printSchema()
df.show(5, truncate=50)

Joined dataset: 1,142,544 rows
root
 |-- user_id: integer (nullable = true)
 |-- book_id: integer (nullable = true)
 |-- is_read: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- date_added: string (nullable = true)



+-------+-------+-------+------+------------------------------+
|user_id|book_id|is_read|rating|                    date_added|
+-------+-------+-------+------+------------------------------+
|      0|    255|      1|     4|Tue Mar 18 09:21:04 -0700 2014|
|      0|    313|      0|     0|Wed May 01 12:55:44 -0700 2013|
|      2|   1171|      1|     3|Mon Mar 17 16:07:36 -0700 2008|
|      2|   1180|      1|     3|Mon Mar 17 16:03:27 -0700 2008|
|      3|   1335|      0|     0|Sat Sep 05 16:42:26 -0700 2015|
+-------+-------+-------+------+------------------------------+
only showing top 5 rows


## 2. Verify data ranges & parse timestamps

In [4]:
# Verify data ranges
print(f"Rating range: {df.select(F.min('rating'), F.max('rating')).collect()[0]}")
print(f"is_read values: {df.select(F.min('is_read'), F.max('is_read')).collect()[0]}")
print(f"Total interactions: {df.count():,} rows")

print("\n=== is_read distribution ===")
df.groupBy("is_read").count().orderBy("is_read").show()
print("=== rating distribution (0=unrated) ===")
df.groupBy("rating").count().orderBy("rating").show()

Rating range: Row(min(rating)=0, max(rating)=5)


is_read values: Row(min(is_read)=0, max(is_read)=1)


Total interactions: 1,142,544 rows

=== is_read distribution ===


+-------+------+
|is_read| count|
+-------+------+
|      0|582915|
|      1|559629|
+-------+------+

=== rating distribution (0=unrated) ===


+------+------+
|rating| count|
+------+------+
|     0|620785|
|     1| 10059|
|     2| 31083|
|     3|116135|
|     4|187434|
|     5|177048|
+------+------+



In [5]:
# Parse date_added: format is "EEE MMM dd HH:mm:ss Z yyyy"
# e.g. "Tue Oct 17 09:40:11 -0700 2017"
df = df.withColumn(
    "date_added_ts",
    F.to_timestamp(F.col("date_added"), "EEE MMM dd HH:mm:ss Z yyyy"),
)

n_parsed = df.filter(F.col("date_added_ts").isNotNull()).count()
n_failed = df.filter(F.col("date_added_ts").isNull()).count()
print(f"Parsed OK:    {n_parsed:,}")
print(f"Parse failed: {n_failed:,}")

# Fallback if the format above doesn't work
if n_failed > n_parsed * 0.5:
    print("Primary format failed — trying unix_timestamp fallback...")
    df = df.drop("date_added_ts").withColumn(
        "date_added_ts",
        F.from_unixtime(
            F.unix_timestamp(F.col("date_added"), "EEE MMM dd HH:mm:ss Z yyyy")
        ).cast("timestamp"),
    )
    n_parsed = df.filter(F.col("date_added_ts").isNotNull()).count()
    n_failed = df.filter(F.col("date_added_ts").isNull()).count()
    print(f"Fallback parsed OK:    {n_parsed:,}")
    print(f"Fallback parse failed: {n_failed:,}")

# Drop rows where timestamp parsing failed
df = df.filter(F.col("date_added_ts").isNotNull())

# Show date range
df.select(
    F.min("date_added_ts").alias("earliest"),
    F.max("date_added_ts").alias("latest"),
).show(truncate=False)

Parsed OK:    1,142,544
Parse failed: 0


+-------------------+-------------------+
|earliest           |latest             |
+-------------------+-------------------+
|1994-01-01 08:00:00|2017-11-04 03:17:20|
+-------------------+-------------------+



## 3. Iterative cold-start filtering

Remove users and books with fewer than 5 interactions (any type). Iterate until stable.

In [6]:
MIN_USER_INTERACTIONS = 5
MIN_BOOK_INTERACTIONS = 5

df_filtered = df.select("user_id", "book_id", "is_read", "rating", "date_added_ts")

spark.sparkContext.setCheckpointDir(f"{GCS_BASE}/data/_checkpoints")

prev_count = 0
curr_count = df_filtered.count()
iteration = 0

while curr_count != prev_count:
    iteration += 1
    prev_count = curr_count

    # Filter users with enough interactions
    user_counts = (
        df_filtered.groupBy("user_id")
        .agg(F.count("*").alias("n"))
        .filter(F.col("n") >= MIN_USER_INTERACTIONS)
        .select("user_id")
    )
    df_filtered = df_filtered.join(user_counts, "user_id", "inner")

    # Filter books with enough interactions
    book_counts = (
        df_filtered.groupBy("book_id")
        .agg(F.count("*").alias("n"))
        .filter(F.col("n") >= MIN_BOOK_INTERACTIONS)
        .select("book_id")
    )
    df_filtered = df_filtered.join(book_counts, "book_id", "inner")

    # Checkpoint every 2 iterations to prevent DAG blowup
    if iteration % 2 == 0:
        df_filtered = df_filtered.checkpoint()

    curr_count = df_filtered.count()
    n_users = df_filtered.select("user_id").distinct().count()
    n_books = df_filtered.select("book_id").distinct().count()
    print(
        f"Iteration {iteration}: {curr_count:,} interactions, "
        f"{n_users:,} users, {n_books:,} books"
    )

print(f"\nFinal filtered dataset: {curr_count:,} interactions")

df_filtered = df_filtered.cache()
df_filtered.count()  # materialize

Iteration 1: 242,636 interactions, 55,187 users, 20,126 books


Iteration 2: 122,009 interactions, 20,797 users, 11,420 books
Iteration 3: 91,606 interactions, 14,811 users, 8,944 books


Iteration 4: 78,451 interactions, 12,479 users, 7,816 books
Iteration 5: 71,292 interactions, 11,259 users, 7,187 books


Iteration 6: 66,764 interactions, 10,509 users, 6,784 books
Iteration 7: 63,742 interactions, 10,023 users, 6,502 books


Iteration 8: 61,619 interactions, 9,674 users, 6,313 books
Iteration 9: 60,100 interactions, 9,426 users, 6,179 books


...

Iteration 22: 53,509 interactions, 8,410 users, 5,539 books
Iteration 23: 53,433 interactions, 8,397 users, 5,533 books


Iteration 24: 53,365 interactions, 8,388 users, 5,525 books
Iteration 25: 53,277 interactions, 8,376 users, 5,515 books


Iteration 26: 53,221 interactions, 8,367 users, 5,510 books
Iteration 27: 53,189 interactions, 8,361 users, 5,508 books


Iteration 28: 53,185 interactions, 8,360 users, 5,508 books
Iteration 29: 53,185 interactions, 8,360 users, 5,508 books

Final filtered dataset: 53,185 interactions


53185

## 4. Temporal train / validation / test split

Split by `date_added_ts` percentiles (by count):
- **Train:** oldest 70%
- **Validation:** next 20% (for hyperparameter tuning)
- **Test:** newest 10% (final held-out evaluation)

In [7]:
# Compute cutoff dates at 70th and 90th percentiles by count
cutoffs = df_filtered.select(
    F.expr(
        "percentile_approx(unix_timestamp(date_added_ts), array(0.7, 0.9))"
    ).alias("cutoffs")
).collect()[0]["cutoffs"]

from datetime import datetime

cutoff_val = datetime.fromtimestamp(cutoffs[0])
cutoff_test = datetime.fromtimestamp(cutoffs[1])

print(f"Train / Val boundary:  {cutoff_val}")
print(f"Val / Test boundary:   {cutoff_test}")

Train / Val boundary:  2015-09-12 14:17:39
Val / Test boundary:   2017-01-02 04:46:03


In [8]:
train = df_filtered.filter(F.col("date_added_ts") < F.lit(cutoff_val))
val = df_filtered.filter(
    (F.col("date_added_ts") >= F.lit(cutoff_val))
    & (F.col("date_added_ts") < F.lit(cutoff_test))
)
test = df_filtered.filter(F.col("date_added_ts") >= F.lit(cutoff_test))

n_train = train.count()
n_val = val.count()
n_test = test.count()
n_all = n_train + n_val + n_test

print(f"Train: {n_train:,} ({n_train/n_all*100:.1f}%)")
print(f"Val:   {n_val:,} ({n_val/n_all*100:.1f}%)")
print(f"Test:  {n_test:,} ({n_test/n_all*100:.1f}%)")

Train: 37,224 (70.0%)
Val:   10,637 (20.0%)
Test:  5,324 (10.0%)


## 5. Post-split filtering

Remove users/books from val and test that don't appear in the training set (ALS cannot score unseen entities).

In [9]:
train_users = train.select("user_id").distinct()
train_books = train.select("book_id").distinct()

# Filter validation set
val_filtered = (
    val
    .join(train_users, "user_id", "inner")
    .join(train_books, "book_id", "inner")
)
n_val_filtered = val_filtered.count()
print(f"Val before filtering: {n_val:,}")
print(f"Val after filtering:  {n_val_filtered:,}")
print(f"Val dropped:          {n_val - n_val_filtered:,} "
      f"({(n_val - n_val_filtered)/n_val*100:.1f}%)")

# Filter test set
test_filtered = (
    test
    .join(train_users, "user_id", "inner")
    .join(train_books, "book_id", "inner")
)
n_test_filtered = test_filtered.count()
print(f"\nTest before filtering: {n_test:,}")
print(f"Test after filtering:  {n_test_filtered:,}")
print(f"Test dropped:          {n_test - n_test_filtered:,} "
      f"({(n_test - n_test_filtered)/n_test*100:.1f}%)")

val = val_filtered
test = test_filtered

Val before filtering: 10,637
Val after filtering:  6,032
Val dropped:          4,605 (43.3%)



Test before filtering: 5,324
Test after filtering:  2,062
Test dropped:          3,262 (61.3%)


## 6. Validation checks

In [10]:
# 1. No temporal leakage
train_max = train.select(F.max("date_added_ts")).collect()[0][0]
val_min = val.select(F.min("date_added_ts")).collect()[0][0]
val_max = val.select(F.max("date_added_ts")).collect()[0][0]
test_min = test.select(F.min("date_added_ts")).collect()[0][0]

print(f"Train max:  {train_max}")
print(f"Val min:    {val_min}")
print(f"Val max:    {val_max}")
print(f"Test min:   {test_min}")

assert train_max <= cutoff_val, "LEAKAGE: train contains dates after val cutoff!"
assert val_max <= cutoff_test, "LEAKAGE: val contains dates after test cutoff!"
print("\nNo temporal leakage detected.")

Train max:  2015-09-12 13:03:18
Val min:    2015-09-12 14:17:39
Val max:    2017-01-02 03:55:03
Test min:   2017-01-02 04:46:03

No temporal leakage detected.


In [11]:
# 2. Coverage statistics
train_users_n = train.select("user_id").distinct().count()
train_books_n = train.select("book_id").distinct().count()
val_users_n = val.select("user_id").distinct().count()
val_books_n = val.select("book_id").distinct().count()
test_users_n = test.select("user_id").distinct().count()
test_books_n = test.select("book_id").distinct().count()

print(f"Train — users: {train_users_n:,}, books: {train_books_n:,}")
print(f"Val   — users: {val_users_n:,}, books: {val_books_n:,}")
print(f"Test  — users: {test_users_n:,}, books: {test_books_n:,}")
print(f"\nVal user coverage:  {val_users_n/train_users_n*100:.1f}% of train users")
print(f"Val book coverage:  {val_books_n/train_books_n*100:.1f}% of train books")
print(f"Test user coverage: {test_users_n/train_users_n*100:.1f}% of train users")
print(f"Test book coverage: {test_books_n/train_books_n*100:.1f}% of train books")

Train — users: 7,615, books: 4,988
Val   — users: 3,400, books: 2,963
Test  — users: 1,528, books: 1,503

Val user coverage:  44.6% of train users
Val book coverage:  59.4% of train books
Test user coverage: 20.1% of train users
Test book coverage: 30.1% of train books


In [12]:
# 3. Rating & is_read distribution comparison
for label, split_df in [("Train", train), ("Val", val), ("Test", test)]:
    print(f"=== {label} rating distribution (0=unrated) ===")
    split_df.groupBy("rating").count().orderBy("rating").show()
    print(f"=== {label} is_read distribution ===")
    split_df.groupBy("is_read").count().orderBy("is_read").show()

# 4. Per-user interaction count percentiles
for label, split_df in [("Train", train), ("Val", val), ("Test", test)]:
    stats = split_df.groupBy("user_id").count()
    print(f"{label} — interactions per user percentiles:")
    stats.select(
        F.expr(
            "percentile_approx(count, array(0.05, 0.25, 0.50, 0.75, 0.95))"
        ).alias("p5_p25_p50_p75_p95")
    ).show(truncate=False)

=== Train rating distribution (0=unrated) ===


+------+-----+
|rating|count|
+------+-----+
|     0|21042|
|     1|  361|
|     2| 1027|
|     3| 3641|
|     4| 5883|
|     5| 5270|
+------+-----+

=== Train is_read distribution ===


+-------+-----+
|is_read|count|
+-------+-----+
|      0|20287|
|      1|16937|
+-------+-----+

=== Val rating distribution (0=unrated) ===


+------+-----+
|rating|count|
+------+-----+
|     0| 4589|
|     1|   30|
|     2|   80|
|     3|  383|
|     4|  530|
|     5|  420|
+------+-----+

=== Val is_read distribution ===


+-------+-----+
|is_read|count|
+-------+-----+
|      0| 4404|
|      1| 1628|
+-------+-----+

=== Test rating distribution (0=unrated) ===


+------+-----+
|rating|count|
+------+-----+
|     0| 1635|
|     1|   13|
|     2|   19|
|     3|  110|
|     4|  171|
|     5|  114|
+------+-----+

=== Test is_read distribution ===


+-------+-----+
|is_read|count|
+-------+-----+
|      0| 1569|
|      1|  493|
+-------+-----+

Train — interactions per user percentiles:


+------------------+
|p5_p25_p50_p75_p95|
+------------------+
|[2, 4, 5, 6, 9]   |
+------------------+

Val — interactions per user percentiles:


+------------------+
|p5_p25_p50_p75_p95|
+------------------+
|[1, 1, 1, 2, 4]   |
+------------------+

Test — interactions per user percentiles:


+------------------+
|p5_p25_p50_p75_p95|
+------------------+
|[1, 1, 1, 1, 3]   |
+------------------+



## 7. Save splits to GCS

In [13]:
# Cast to final schema
# NOTE: r = 1 + 2*is_read + beta*max(0, rating-3) is computed in notebook 03
# because beta is a tunable hyperparameter.
train_als = train.select(
    F.col("user_id").cast("int"),
    F.col("book_id").cast("int"),
    F.col("is_read").cast("int"),
    F.col("rating").cast("int"),
)
val_als = val.select(
    F.col("user_id").cast("int"),
    F.col("book_id").cast("int"),
    F.col("is_read").cast("int"),
    F.col("rating").cast("int"),
)
test_als = test.select(
    F.col("user_id").cast("int"),
    F.col("book_id").cast("int"),
    F.col("is_read").cast("int"),
    F.col("rating").cast("int"),
)

train_als.printSchema()
train_als.show(5)

root
 |-- user_id: integer (nullable = true)
 |-- book_id: integer (nullable = true)
 |-- is_read: integer (nullable = true)
 |-- rating: integer (nullable = true)

+-------+-------+-------+------+
|user_id|book_id|is_read|rating|
+-------+-------+-------+------+
| 291417|  14832|      1|     3|
| 387996|   6466|      0|     0|
| 142263|  17679|      1|     3|
| 224425|   1591|      0|     0|
|  80718|   1959|      0|     0|
+-------+-------+-------+------+
only showing top 5 rows


In [14]:
train_als.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/train")
val_als.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/val")
test_als.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/test")

print("Saved splits to GCS.")

# Verify by reading back
train_check = spark.read.parquet(f"{OUTPUT_BASE}/train")
val_check = spark.read.parquet(f"{OUTPUT_BASE}/val")
test_check = spark.read.parquet(f"{OUTPUT_BASE}/test")

print(f"Train: {train_check.count():,} rows")
print(f"Val:   {val_check.count():,} rows")
print(f"Test:  {test_check.count():,} rows")

Saved splits to GCS.


Train: 37,224 rows


Val:   6,032 rows


Test:  2,062 rows


In [15]:
print("=" * 60)
print("  DATA SPLIT SUMMARY")
print("=" * 60)
print(f"Timestamp column:       date_added")
print(f"Train/Val cutoff:       {cutoff_val}")
print(f"Val/Test cutoff:        {cutoff_test}")
print(f"Min user interactions:  {MIN_USER_INTERACTIONS}")
print(f"Min book interactions:  {MIN_BOOK_INTERACTIONS}")
print(f"")
print(f"Train — {n_train:>12,} interactions, {train_users_n:>9,} users, {train_books_n:>9,} books")
print(f"Val   — {n_val_filtered:>12,} interactions, {val_users_n:>9,} users, {val_books_n:>9,} books")
print(f"Test  — {n_test_filtered:>12,} interactions, {test_users_n:>9,} users, {test_books_n:>9,} books")
print(f"")
print(f"Output schema: (user_id: int, book_id: int, is_read: int, rating: int)")
print(f"Output paths:")
print(f"  Train: {OUTPUT_BASE}/train")
print(f"  Val:   {OUTPUT_BASE}/val")
print(f"  Test:  {OUTPUT_BASE}/test")

  DATA SPLIT SUMMARY
Timestamp column:       date_added
Train/Val cutoff:       2015-09-12 14:17:39
Val/Test cutoff:        2017-01-02 04:46:03
Min user interactions:  5
Min book interactions:  5

Train —       37,224 interactions,     7,615 users,     4,988 books
Val   —        6,032 interactions,     3,400 users,     2,963 books
Test  —        2,062 interactions,     1,528 users,     1,503 books

Output schema: (user_id: int, book_id: int, is_read: int, rating: int)
Output paths:
  Train: gs://nshen7-personal-bucket/projects/rec_sys_goodreads/data/splits_sample_mode/train
  Val:   gs://nshen7-personal-bucket/projects/rec_sys_goodreads/data/splits_sample_mode/val
  Test:  gs://nshen7-personal-bucket/projects/rec_sys_goodreads/data/splits_sample_mode/test
